# MDA-MB-468 — CyTOFSTD ingestion and comparison of three `cytof_transform` normalizations

**Data:** `Data/MDAMB/MDAMB468_Yael_cols_corrected.csv` — 67,563 cells x 32 markers, **raw (linear, pre-arcsinh) intensities**,
extracted from `CyTOF_Data/CyTOF1_MDAMB468_Yael.fcs`.

**Goal:** ingest with `cytofstandard`, apply three normalization variants from `cytof_transform` 0.2.0 to the
*same* QC'd cells, and decide which is best.

| label | call | technical factor | correction |
|---|---|---|---|
| **divide** (B1) | `method="divide"` | `M` = std-minimizing convex-weighted mean of mean-normalized core histones | `x / M` on **raw** counts, **then** `arcsinh(x/5)` |
| **regress single-γ** (A) | `method="regress"`, `gamma_mode="single"` | `f` = PC1 of core histones | `x − γ̄·(f − med f)`, one shared γ̄ for every marker |
| **regress per-marker γ** (A) | `method="regress"`, `gamma_mode="per_marker"` | `f` = PC1 of core histones | `x − γ_m·(f − med f)`, γ_m fitted per marker |

These three are not an arbitrary set — they factor the design along two independent axes:

- **divide vs regress single-γ** — same *uniform* correction applied to every marker, but multiplicative on raw
  counts vs additive in arcsinh space. Isolates the **form and scale** of the correction.
- **regress single-γ vs regress per-marker γ** — same additive arcsinh form, but one shared slope vs a slope
  fitted per marker. Isolates **uniform vs per-marker sensitivity**.

**A fourth arm was tested and rejected — do not re-add it.** `protect_covariates=["KI67","H3S28p"]` fits each
marker on `[f, KI67, H3S28p]` jointly and subtracts only the `f` term, the intent being to spare the
proliferation axis. On MDA-MB-468 it shrank γ for 92% of markers (mean −0.195) and drove several **negative**
(H3K27ac 0.479→−0.021, H2AK119ub 0.155→−0.123), which is physically meaningless: γ<0 means the correction
*adds* signal to cells that stained more. The cause is collinearity — KI67 and H3S28p themselves correlate
0.51 and 0.60 with the technical factor, so conditioning on them collapses the `f` coefficient panel-wide.
The result is systematic under-correction that superficially reads as biology preservation. This is also why
the covariates were a bad choice on their own terms: KI67 and H3S28p encode cell-cycle **biology**, not
permeability.

**Design decisions**
- QC is a single floor: core histones `H3`, `H4`, `H3.3` all `> 5` on raw data. Everything else was handled upstream.
- `control_markers` = the three core histones. `markers_to_correct` = the **25 intracellular** non-core-histone
  markers, identical across methods. The 4 surface markers (`CD24`, `CD44`, `CD49f`, `EpCAM`) are **not**
  corrected — permeabilization affects intracellular staining only — which also makes them a negative control
  (see section 4).
- After normalization the **core histones are dropped** and never used again.
- UMAP + Leiden are built on the **16 epigenetic modifications only**; marker plots are shown for **all** markers.

## 0. Setup

In [ ]:
import sys, shutil, warnings, json
from pathlib import Path

# cytof_transform is not pip-installed; it is used via sys.path injection.
# NOTE: deliberately do NOT add /Code to sys.path. If `CyTOFHelper` becomes importable,
# cytof_transform._cytof_divide_b1 silently swaps in the legacy normalize_data and the
# reported permeabilization factor stops matching the applied correction.
CT_PATH = "/Users/ronguy/Dropbox/Work/CyTOF/Code/cytof-transform"
if CT_PATH not in sys.path:
    sys.path.insert(0, CT_PATH)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as ss

import scanpy as sc

import cytof_transform
from cytofstandard import Project

warnings.filterwarnings("ignore", category=UserWarning, module="umap")
sc.settings.verbosity = 1
%matplotlib inline

plt.rcParams.update({
    'axes.labelsize': 11, 'axes.titlesize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'figure.dpi': 110, 'pdf.fonttype': 42, 'ps.fonttype': 42,
})
sns.set_style("white")

print("cytof_transform", cytof_transform.__version__, "->", cytof_transform.__file__)
try:
    import CyTOFHelper  # noqa: F401
    print("!! WARNING: CyTOFHelper is importable — the divide path will use the LEGACY "
          "implementation, not the native std-min divide documented in this notebook.")
except ModuleNotFoundError:
    print("CyTOFHelper not importable -> native std-min divide will be used (intended).")

In [ ]:
def uns_history(adata, key):
    """Read uns[key]['history'] as a list of dicts.

    AnnData round-trips these records through zarr as JSON *strings*, so a freshly
    written history holds dicts but a reloaded one holds str. Normalize both.
    """
    node = adata.uns.get(key, None)
    if node is None or not hasattr(node, "get"):
        return []
    hist = node.get("history", None)
    if hist is None:
        return []
    # NB: `hist` round-trips as a numpy array, so never use `or []` / truthiness on it.
    out = []
    for e in list(hist):
        if isinstance(e, (str, bytes)):
            try:
                e = json.loads(e)
            except Exception:
                continue
        if isinstance(e, dict):
            out.append(e)
    return out

In [ ]:
LINE       = "MDAMB468"
LINE_DISP  = "MDA-MB-468"

BASE       = Path("/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina")
DATA_CSV   = BASE / "Data/MDAMB/MDAMB468_Yael_cols_corrected.csv"
PLOTS      = BASE / "Plots"; PLOTS.mkdir(exist_ok=True)

REG          = Path("/Users/ronguy/Dropbox/Work/CyTOF/Code/CyTOFSTD/cytof_marker_registry_files")
PROJECT_PATH = Path(f"/Users/ronguy/Dropbox/Work/CyTOF/Projects/{LINE}_NormCompare")
RUN_ID       = "MDAMB468"

In [ ]:
ARCSINH_COFACTOR = 5.0
SEED             = 42
REBUILD          = False   # set True to wipe the project and re-run everything from scratch

# ---- the three normalization variants -------------------------------------------------
# key -> (display label, colour, kwargs for normalize_with_cytof_transform)
METHODS = {
    "divide":  ("divide (B1)",            "#3f78c1", dict(method="divide")),
    "single":  ("regress, single γ",      "#33a02c", dict(method="regress", gamma_mode="single")),
    "regress": ("regress, per-marker γ",  "#e2725b", dict(method="regress", gamma_mode="per_marker")),
}
MKEYS  = list(METHODS)
LABEL  = {k: v[0] for k, v in METHODS.items()}
COLOR  = {k: v[1] for k, v in METHODS.items()}

LAYER_OF = {k: f"norm_{k}"    for k in MKEYS}   # corrected, arcsinh scale
ZLAYER_OF= {k: f"norm_{k}_z"  for k in MKEYS}   # z from cytof_transform (corrected markers only)
ZS_OF    = {k: f"norm_{k}_zs" for k in MKEYS}   # explicit z-score step (section 5.3)
EMB_OF   = {k: f"umap_{k}"   for k in MKEYS}
CLKEY_OF = {k: f"cl_{k}"     for k in MKEYS}
TF_OF    = {k: f"tf_{k}"     for k in MKEYS}

## 1. Raw data inspection

In [ ]:
raw_df = pd.read_csv(DATA_CSV)
print(f"{raw_df.shape[0]:,} cells x {raw_df.shape[1]} markers")
print(f"NaNs: {raw_df.isna().sum().sum()}   negatives: {(raw_df < 0).sum().sum()}   zeros: {(raw_df == 0).sum().sum():,}")
raw_df.describe().T[['min', '25%', '50%', '75%', 'max']].round(2)

Values are large positive intensities with a hard floor at 0 and no negatives — this is **raw, pre-arcsinh**
data, which is exactly what the `divide` method requires. There are no DNA, Event-length or bead channels and
only one sample, so classical doublet/bead QC is not available here (and per instruction was done upstream).

In [ ]:
from tqdm.notebook import tqdm

## 2. Ingest into a CyTOFSTD project

In [ ]:
if REBUILD and PROJECT_PATH.exists():
    shutil.rmtree(PROJECT_PATH)

try:
    project = Project.load(str(PROJECT_PATH))
    print(f"Loaded existing project at {PROJECT_PATH}")
except Exception:
    PROJECT_PATH.parent.mkdir(parents=True, exist_ok=True)
    project = Project.create(
        str(PROJECT_PATH),
        project_id=f"{LINE}_NormCompare",
        project_name=f"{LINE_DISP} normalization method comparison",
        standard_marker_file=str(REG / "standard_markers.csv"),
        marker_alias_file=str(REG / "marker_aliases.yaml"),
    )
    print(f"Created project at {PROJECT_PATH}")

In [ ]:
# cytofstandard's ingest needs a sample-metadata table with file_name / sample_id / line_id.
PREP = PROJECT_PATH / "preprocessed"; PREP.mkdir(parents=True, exist_ok=True)
meta_path = PREP / "samples.csv"
pd.DataFrame([{
    "file_name": DATA_CSV.name,
    "sample_id": RUN_ID,
    "line_id":   RUN_ID,
    "condition": "baseline",
}]).to_csv(meta_path, index=False)

if project.has_run(RUN_ID):
    run = project.get_run(RUN_ID)
    print(f"Run '{RUN_ID}' already exists — skipping ingestion.")
else:
    run = project.add_run(RUN_ID, run_name=f"{LINE_DISP} (cols corrected)")
    run.ingest(
        files=[str(DATA_CSV)],
        sample_metadata=str(meta_path),
        strict_markers=False,
        allow_extra_markers=True,
        show_marker_coverage=True,
    )
    print("Ingested.")

adata = run.read_adata()
print(f"\nAnnData: {adata.n_obs:,} cells x {adata.n_vars} markers")
print("X is raw (identical to layers['raw']):", np.allclose(adata.X, adata.layers['raw']))

In [ ]:
# All 32 channels resolve exactly against the standard marker registry.
run.ingestion_summary()

## 3. QC — core histones > 5

The only QC applied: a cell must have `H3 > 5`, `H4 > 5` and `H3.3 > 5` on raw data.
`qc_gate` bounds are inclusive (`>=`), but no cell sits exactly at 5, so `lower=5` is identical to `> 5` here.

In [ ]:
CORE_HISTONES = run.markers_core_histones           # from the registry: is_core_histone == True
print("core histones (registry):", CORE_HISTONES)

pre = run.read_adata()
raw_pre = pd.DataFrame(pre.layers["raw"], columns=pre.var_names, index=pre.obs_names)

for m in CORE_HISTONES:
    n_bad = int((raw_pre[m] <= 5).sum())
    print(f"  {m:6s}  <=5 : {n_bad:5d} cells ({100*n_bad/len(raw_pre):.2f}%)")
print(f"  exactly ==5 anywhere: {int((raw_pre[CORE_HISTONES] == 5).sum().sum())}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, m in zip(axes, CORE_HISTONES):
    ax.hist(np.log10(raw_pre[m] + 1), bins=200, color="0.35")
    ax.axvline(np.log10(6), color="crimson", lw=1.5, ls="--", label="gate (>5)")
    ax.set_title(m); ax.set_xlabel("log10(raw + 1)"); ax.set_yscale("log")
    ax.legend(fontsize=8)
axes[0].set_ylabel("cells")
fig.suptitle("QC gate on core histones (raw scale)", y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
qc_hist = uns_history(run.read_adata(), "qc")
if qc_hist:
    h = qc_hist[-1]
    print(f"QC already applied ({len(qc_hist)} event(s)) — skipping to stay idempotent.")
    print(f"  n_before={h['n_before']:,}  n_after={h['n_after']:,}  n_removed={h['n_removed']:,}")
else:
    mask = run.qc_gate({m: {"lower": 5} for m in CORE_HISTONES}, layer="raw", inplace=True)
    print(f"QC: kept {int(mask.sum()):,} / {len(mask):,} cells  (removed {int((~mask).sum()):,}, "
          f"{100*(~mask).mean():.2f}%)")

adata = run.read_adata()
print(f"\nPost-QC: {adata.n_obs:,} cells x {adata.n_vars} markers")

## 4. Marker groups

Permeabilization is a property of getting antibody **inside** the cell, so only **intracellular** markers are
corrected. The four extracellular markers (`CD24`, `CD44`, `CD49f`, `EpCAM`, taken from the registry's
`intra_extra` field) are deliberately left out of `markers_to_correct`.

This matters for two reasons:

1. **Correctness.** A surface marker has no mechanistic dependence on permeabilization. Applying a correction
   to it can only *inject* the technical factor into a channel that never carried it.
2. **They become a built-in negative control.** Because they pass through untouched, any residual correlation
   between a surface marker and `tech_ref` after normalization is, by construction, **not** something
   normalization failed to remove — it is evidence that `tech_ref` itself carries biology (cell size, ploidy,
   cell-cycle stage). That is used in section 9 to calibrate how much of the "technical" axis is really
   technical.

In [ ]:
ALL_MARKERS = run.read_adata().var_names.tolist()

# 16 histone post-translational modifications -> drive UMAP + clustering.
# EZH2 is registry class "Chromatin" but is a writer enzyme, not a modification, so it is
# excluded from the clustering feature set (kept for plotting).
EPI = [m for m in ['H2AK119ub','H3K27ac','H3K27me2','H3K27me3','H3K36me2','H3K36me3',
                   'H3K4me1','H3K4me3','H3K64ac','H3K9ac','H3K9me2','H3K9me3',
                   'H3S28p','H4K16ac','H4K20me3','pH2A.X'] if m in ALL_MARKERS]

# Permeabilization affects INTRACELLULAR staining. Surface markers have no mechanistic
# reason to depend on it, so they are excluded from the correction entirely — correcting
# them can only inject the technical factor into channels that never carried it.
SURFACE = [m for m in run.markers_extracellular if m in ALL_MARKERS]
INTRA   = [m for m in run.markers_intracellular if m in ALL_MARKERS]

TO_CORRECT   = [m for m in INTRA if m not in CORE_HISTONES]   # 25 intracellular markers
INTRA_OTHER  = [m for m in TO_CORRECT if m not in EPI]        # intracellular, non-epigenetic

# post-normalization marker universe (core histones dropped, per instruction)
MARKERS_POST = [m for m in ALL_MARKERS if m not in CORE_HISTONES]
OTHER = [m for m in MARKERS_POST if m not in EPI]

# marker -> role, used to group every downstream metric
ROLE = {m: ("epigenetic" if m in EPI else
            "intracellular (other)" if m in INTRA_OTHER else
            "surface (UNcorrected)") for m in MARKERS_POST}

print(f"CORE_HISTONES ({len(CORE_HISTONES):2d}) control markers   : {CORE_HISTONES}")
print(f"TO_CORRECT    ({len(TO_CORRECT):2d}) corrected          : intracellular only")
print(f"  EPI         ({len(EPI):2d}) -> UMAP/clustering : {EPI}")
print(f"  INTRA_OTHER ({len(INTRA_OTHER):2d})                    : {INTRA_OTHER}")
print(f"SURFACE       ({len(SURFACE):2d}) NOT corrected      : {SURFACE}")

## 5. Normalization — three variants

All three start from the **same** post-QC `raw` layer and correct the **same** 29 markers, writing to separate
layers so nothing is overwritten.

`normalize_with_cytof_transform` writes the tech factor to `obs["norm_tech_factor"]` on every call, so it is
copied to a method-specific column immediately after each one.

Note on the fourth available mode: `gamma_mode="shrink"` is **not** included, and deliberately. Its shrinkage
weight is `B = τ²/(τ² + SE²)`; with ~67k cells `SE → 0`, so `B → 1` and it collapses back onto `per_marker`,
adding a redundant arm. `"shrink_stability"` would need a stability grouping variable, which a single sample
does not provide.

In [ ]:
norm_hist  = uns_history(run.read_adata(), "normalization")
# key on corrected_layer, NOT on method: two of the three arms share method="regress"
done_layers = {h.get("corrected_layer") for h in norm_hist}
print("normalized layers already present:", done_layers or "none")

In [ ]:
summaries = {}
for k in MKEYS:
    layer = LAYER_OF[k]
    if layer in done_layers:
        summaries[k] = [h for h in norm_hist if h.get("corrected_layer") == layer][-1]
        print(f"{LABEL[k]:24s} : already applied — skipping.")
        continue
    print(f"{LABEL[k]:24s} : running ...")
    summaries[k] = run.normalize_with_cytof_transform(
        control_markers=CORE_HISTONES,
        markers_to_correct=TO_CORRECT,
        source_layer="raw",
        input_is_arcsinh=False,          # divide: raw->divide->arcsinh; regress: wrapper arcsinhs first
        arcsinh_cofactor=ARCSINH_COFACTOR,
        groupby_col="sample_id",
        corrected_layer=layer,
        z_layer=ZLAYER_OF[k],
        anchor_to_median=True,
        zscore=True,
        **METHODS[k][2],
    )
    a = run.read_adata(); a.obs[TF_OF[k]] = a.obs["norm_tech_factor"].values; run.save(a)

pd.DataFrame({k: {"method": s["method"], "gamma_mode": s["gamma_mode"],
                  "tech_factor": s["tech_factor_kind"], "layer": s["corrected_layer"]}
              for k, s in summaries.items()}).T

In [ ]:
adata = run.read_adata()

raw_all   = pd.DataFrame(adata.layers["raw"], columns=adata.var_names, index=adata.obs_names)
asinh_raw = np.arcsinh(raw_all / ARCSINH_COFACTOR)          # uncorrected reference, arcsinh scale

def layer_df(layer, markers=MARKERS_POST):
    M = pd.DataFrame(adata.layers[layer], columns=adata.var_names, index=adata.obs_names)
    return M[markers]

NORM = {k: layer_df(LAYER_OF[k]) for k in MKEYS}    # all markers, core histones dropped
# NORMz (the z-scored epigenetic matrix used for UMAP/clustering) is built in section 5.1,
# after the explicit z-scoring pass.

print("layers:", sorted(adata.layers.keys()))
print()
print(f"{'':26s} {'min':>8s} {'max':>8s} {'mean':>8s}")
print(f"{'raw (arcsinh)':26s} {asinh_raw[MARKERS_POST].values.min():8.3f} "
      f"{asinh_raw[MARKERS_POST].values.max():8.3f} {asinh_raw[MARKERS_POST].values.mean():8.3f}")
for k in MKEYS:
    M = NORM[k].values
    print(f"{LABEL[k]:26s} {M.min():8.3f} {M.max():8.3f} {M.mean():8.3f}")

### 5.1 Explicit z-scoring of every method

`cytof_transform` already emits a z-scored layer (`zscore=True`), but it only z-scores the markers listed in
`markers_to_correct` and does it inside the transform. An explicit pass is applied here instead so that the
step is visible, provenance-logged, and **identical across all three methods** — every marker is centred and
scaled the same way, so nothing downstream can be attributed to a difference in scaling.

`zscore_markers_balanced` z-scores within `groupby_col` on a group-balanced subsample; with a single sample
this reduces to a plain per-marker z-score, but it is the right call to use for provenance and it generalizes
if more samples are added later.

Note the shipped default `source_layer="normlized"` is misspelled in the package, so it is always passed
explicitly here.

In [ ]:
adata = run.read_adata()
for k in MKEYS:
    if ZS_OF[k] in adata.layers:
        print(f"{LABEL[k]:24s} : '{ZS_OF[k]}' exists — skipping.")
        continue
    run.zscore_markers_balanced(
        source_layer=LAYER_OF[k],      # NB: package default is misspelled "normlized"
        output_layer=ZS_OF[k],
        groupby_col="sample_id",
        random_state=SEED,
    )
    print(f"{LABEL[k]:24s} : z-scored {LAYER_OF[k]} -> {ZS_OF[k]}")

adata = run.read_adata()
chk = pd.DataFrame({
    LABEL[k]: pd.DataFrame(adata.layers[ZS_OF[k]], columns=adata.var_names)[EPI]
              .agg(['mean', 'std']).T.stack()
    for k in MKEYS
}).round(4)
print("\nper-marker mean/std of the epigenetic marks after z-scoring (should be ~0 / ~1):")
chk.groupby(level=1).agg(['min', 'max']).round(4)

In [ ]:
# Explicitly z-scored frames.
#   NORMZS : all markers (core histones dropped) -> used for the UMAP marker plots
#   NORMz  : epigenetic marks only               -> drives UMAP + Leiden
NORMZS = {k: pd.DataFrame(adata.layers[ZS_OF[k]], columns=adata.var_names,
                          index=adata.obs_names)[MARKERS_POST] for k in MKEYS}
NORMz  = {k: NORMZS[k][EPI] for k in MKEYS}
print({k: (NORMZS[k].shape, NORMz[k].shape) for k in MKEYS})

This plot is the crux of the comparison. `divide` and `regress single-γ` both apply **one** correction strength
to every marker — the dashed line. `regress per-marker γ` lets each marker have its own. Markers whose bar sits
far from the dashed line are the ones the two uniform methods necessarily get wrong: under-corrected where
`γ_m > γ̄`, over-corrected where `γ_m < γ̄`, and actively damaged where `γ_m ≈ 0` but a full correction is applied
anyway.

## 6. UMAP and clustering on the epigenetic modifications

UMAP and Leiden use the **16 epigenetic marks only**, from each method's z-scored layer (all three z-score the
same 16 markers, so the inputs are on a comparable footing).

In [ ]:
MKEYS=['single']

In [ ]:
UMAP_KW = dict(n_neighbors=15, min_dist=0.1, metric="euclidean", random_state=SEED)
RESOLUTION = .15

existing_emb = set((run.read_adata().uns.get("embeddings", {}) or {}).keys())
for k in MKEYS:
    if EMB_OF[k] in existing_emb:
        print(f"{LABEL[k]:24s} : embedding exists — skipping UMAP.")
    else:
        print(f"{LABEL[k]:24s} : computing UMAP on {len(EPI)} epigenetic marks ...")
        run.compute_umap(markers=EPI, source_layer=ZS_OF[k],
                         embedding_name=EMB_OF[k], **UMAP_KW)
    run.cluster_leiden(embedding_name=EMB_OF[k], cluster_key=CLKEY_OF[k],
                       resolution=RESOLUTION, seed=SEED)

adata = run.read_adata()
for k in MKEYS:
    print(f"{LABEL[k]:24s} -> {adata.obs[CLKEY_OF[k]].nunique()} Leiden clusters")

In [ ]:
adata = run.read_adata()
U  = {k: np.asarray(adata.obsm[EMB_OF[k]])  for k in MKEYS}
CL = {k: adata.obs[CLKEY_OF[k]].astype(str) for k in MKEYS}

# scanpy resolves `basis` against obsm directly, so EMB_OF names work as-is.
# Cluster keys must be categorical for a discrete palette.
for k in MKEYS:
    adata.obs[CLKEY_OF[k]] = adata.obs[CLKEY_OF[k]].astype(str).astype("category")

fig, axes = plt.subplots(1, len(MKEYS), figsize=(6.3 * len(MKEYS), 5.8))
for ax, k in zip(np.atleast_1d(axes).ravel(), MKEYS):
    sc.pl.embedding(
        adata, basis=EMB_OF[k], color=CLKEY_OF[k], ax=ax, show=False,
        legend_loc="on data", legend_fontsize=9, legend_fontoutline=2,
        palette="tab20", size=3, frameon=False, use_raw=False,
        title=f"{LABEL[k]} — {CL[k].nunique()} clusters",
    )
fig.suptitle("UMAP on the 16 epigenetic modifications", y=1.02, fontsize=13)
plt.tight_layout(); plt.savefig(PLOTS / f"{LINE}_norm_umap_clusters.png", dpi=180, bbox_inches="tight"); plt.show()

## 7. All markers on every UMAP

In [ ]:
def marker_grid(k, markers, ncols=5, fname=None):
    """UMAP coloured by each z-scored marker, via scanpy, with large bold visible text."""
    nrows = int(np.ceil(len(markers) / ncols))
    fig, axes_arr = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.6))
    axes_flat = axes_arr.ravel()

    for idx, mname in enumerate(markers):
        ax = axes_flat[idx]
        sc.pl.embedding(
            adata, basis=EMB_OF[k], color=mname, layer=ZS_OF[k],
            cmap="seismic", vcenter=0, vmin="p1", vmax="p99",
            ax=ax, size=4, frameon=False, use_raw=False,
            colorbar_loc="right", show=False
        )
        ax.set_title(mname, fontsize=13, fontweight="bold", pad=6)

    for ax in axes_flat[len(markers):]:
        ax.axis("off")

    fig.suptitle(f"{LINE_DISP} — {LABEL[k]} — All Markers (Z-Scored, Seismic)",
                 fontsize=18, fontweight="bold", y=1.02)
    plt.tight_layout()
    if fname:
        fig.savefig(PLOTS / fname, dpi=200, bbox_inches="tight")
    plt.show()

ALL_MARKERS_SORTED = sorted(EPI + OTHER, key=lambda s: s.lower())


In [ ]:
marker_grid("single", ALL_MARKERS_SORTED, fname=f"{LINE}_umap_single_allmarkers.png")

### 7.1 Cluster marker profiles

In [ ]:
NONEPI=['CD24',
 'CD44',
 'CD49f',
 'ER',
 'EZH2',
 'EpCAM',
 'GATA3',
 'KI67',
 'KRT5',
 'KRT8-18',
 'Vimentin',
 'ZEB1',
 'p53',
]

In [ ]:
run.plot_heatmap(EPI,CLKEY_OF['single'],layer=ZS_OF['single'],center=0)
plt.savefig("Plots/MDMAB4680_HM1.png",dpi=200,bbox_inches='tight')
run.plot_heatmap(NONEPI,CLKEY_OF['single'],layer=ZS_OF['single'],center=0)
plt.savefig("Plots/MDMAB4680_HM2.png",dpi=200,bbox_inches='tight')

In [ ]:
adata = run.read_adata()
nh = uns_history(adata, "normalization")
prov = pd.DataFrame([{
    "method": h["method"], "gamma_mode": h["gamma_mode"],
    "tech_factor_kind": h["tech_factor_kind"], "corrected_layer": h["corrected_layer"],
    "module_version": h["module_version"], "entry_point": h["entry_point"],
    "arcsinh_cofactor": h["arcsinh_cofactor"], "n_cells": h["n_after"],
} for h in nh])
print(f"project : {PROJECT_PATH}")
print(f"run     : {RUN_ID}   ({adata.n_obs:,} cells x {adata.n_vars} markers)")
print(f"layers  : {sorted(adata.layers.keys())}")
print(f"plots   : {PLOTS}")
prov

## 10. Pairwise 2D Raw Scatterplots Colored by Single-γ Leiden Clusters
Generates log10 2D scatterplots for all marker pairs using **raw counts**, with cells colored by their **Leiden cluster ID** derived from normalized z-scored data. Saved into `Plots/MDAMB468_Raw_Pairwise_Scatter_Clusters/`.

In [ ]:
import itertools

PAIRWISE_DIR = PLOTS / f"{LINE}_Raw_Pairwise_Scatter_Clusters"
PAIRWISE_DIR.mkdir(parents=True, exist_ok=True)

adata = run.read_adata()
raw_df = pd.DataFrame(adata.layers["raw"], columns=adata.var_names, index=adata.obs_names)
cluster_key = CLKEY_OF["single"]
cluster_series = adata.obs[cluster_key].astype(str).astype("category")

categories = list(cluster_series.cat.categories)
cmap = plt.get_cmap("tab20")
cluster_colors = {c: cmap(i % 20) for i, c in enumerate(categories)}

markers = [m for m in adata.var_names.tolist() if m not in CORE_HISTONES]
pairs = list(itertools.combinations(markers, 2))

print(f"Generating {len(pairs)} pairwise 2D raw log-log scatterplots colored by Leiden clusters in {PAIRWISE_DIR} ...")

for m1, m2 in tqdm(pairs):
    fname = PAIRWISE_DIR / f"{m1.replace('/', '_')}_vs_{m2.replace('/', '_')}_raw_clusters.png"
    if fname.exists():
        continue
    
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    x = np.log10(raw_df[m1] + 1)
    y = np.log10(raw_df[m2] + 1)
    
    for cat in categories:
        mask = cluster_series == cat
        ax.scatter(x[mask], y[mask], s=3, color=cluster_colors[cat], alpha=0.6, label=f"Cluster {cat}")
    
    ax.set_xlabel(f"log10(raw {m1} + 1)")
    ax.set_ylabel(f"log10(raw {m2} + 1)")
    ax.set_title(f"{m1} vs {m2} (Raw Data — Colored by Single-γ Leiden Clusters)", fontsize=10, fontweight="bold")
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=8, markerscale=2)
    plt.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Completed saving {len(pairs)} pairwise scatterplots to {PAIRWISE_DIR}.")
